# Analyse de données de la graphe 

## Import libraries

In [ ]:
import pandas as pd
from pathlib import Path

## Read CSV file to be analysed

In [ ]:
path = Path("../resources/graph3")
csv_path = path / "FDS_COMEXVIANDE_1999.csv"

In [ ]:
df = pd.read_csv(csv_path, sep=";", decimal=",", dtype={
    "N053_MOD": str,
    "N500_MOD": str,
})
df.head()

# Check mixed types of columns

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
cols = range(len(df.columns))
for idx in cols:
    col_name = df.columns[idx]
    type_counts = df.iloc[:, idx].apply(type).value_counts()
    print(type_counts, "\n")

## Analyse different types of value in each column

### Trade per country 

In [ ]:
print(df["COMEXVIANDE_DIM2_LIB"].value_counts().to_string())

### List of possible values in each column with the frequency

In [ ]:
for col in df.columns:
    print(df[col].value_counts(), "\n")

## Transform periodicity (months) to integer

In [ ]:
df["N053_MOD"] = df["N053_MOD"].str.split('.').str[-1].astype(int)

## Verify mapping of label and value

### Country

In [ ]:
country_pairs = df[["COMEXVIANDE_DIM2_LIB", "COMEXVIANDE_DIM2_MOD"]].drop_duplicates()
is_one_to_one = (
    len(country_pairs) == df["COMEXVIANDE_DIM2_LIB"].nunique(dropna=False)
    and len(country_pairs) == df["COMEXVIANDE_DIM2_MOD"].nunique(dropna=False)
)
print("Strict 1-to-1 pays d'echange:", is_one_to_one)

pays_mapper = dict(zip(country_pairs["COMEXVIANDE_DIM2_MOD"].to_list(), country_pairs["COMEXVIANDE_DIM2_LIB"].to_list()))
pays_mapper

### Period

In [ ]:
periodicite_pairs = df[["N053_LIB", "N053_MOD"]].drop_duplicates()
is_one_to_one = (
    len(periodicite_pairs) == df["N053_LIB"].nunique(dropna=False)
    and len(periodicite_pairs) == df["N053_MOD"].nunique(dropna=False)
)
print("Strict 1-to-1 periodicite:", is_one_to_one)

periodicite_mapper = dict(zip(periodicite_pairs["N053_MOD"].to_list(), periodicite_pairs["N053_LIB"].to_list()))
periodicite_mapper

### Indicator (Export, import, balance) in tonne or euro

In [ ]:
indicateur_pairs = df[["COMEXVIANDE_DIM1_LIB", "COMEXVIANDE_DIM1_MOD"]].drop_duplicates()
is_one_to_one = (
    len(indicateur_pairs) == df["COMEXVIANDE_DIM1_LIB"].nunique(dropna=False)
    and len(indicateur_pairs) == df["COMEXVIANDE_DIM1_MOD"].nunique(dropna=False)
)
print("Strict 1-to-1 indicateur:", is_one_to_one)

indicateur_mapper = dict(zip(indicateur_pairs["COMEXVIANDE_DIM1_MOD"].to_list(), indicateur_pairs["COMEXVIANDE_DIM1_LIB"].to_list()))
indicateur_mapper

### Type of meat

In [ ]:
type_de_viande_pairs = df[["N500_LIB", "N500_MOD"]].drop_duplicates()
is_one_to_one = (
    len(type_de_viande_pairs) == df["N500_LIB"].nunique(dropna=False)
    and len(type_de_viande_pairs) == df["N500_MOD"].nunique(dropna=False)
)
print("Strict 1-to-1 type de viande:", is_one_to_one)

type_de_viande_mapper = dict(zip(type_de_viande_pairs["N500_MOD"].to_list(), type_de_viande_pairs["N500_LIB"].to_list()))
type_de_viande_mapper

## Rename columns name

In [ ]:
df = df.rename(columns={
    "ANNREF": "annee",
    "COMEXVIANDE_DIM2_MOD": "pays",
    "N500_MOD": "type",
    "VALEUR": "valeur"
})

In [ ]:
df.head()

In [ ]:
df = df[(df["N053_MOD"] == 0) & (df["COMEXVIANDE_DIM1_MOD"] == 2)]
df

## Save mapper to json

In [ ]:
import json

In [ ]:
mappers = {
    "pays": pays_mapper,
    "periodicite": periodicite_mapper,
    "indicateur": indicateur_mapper,
    "type_viande": type_de_viande_mapper
}
# preview of json result
print(json.dumps(mappers, indent=4, ensure_ascii=False))

In [ ]:
destination = "mapping_rules.json"
with open(path / destination, "w", encoding="utf-8") as file:
   json.dump(mappers, file, indent=4, ensure_ascii=False)

In [ ]:
with open(path / destination, "r") as file:
    mapper = json.load(file)
mapper

## Save dataframe to csv

In [ ]:
df.to_csv(csv_path, sep=",", decimal=".", index=False, columns=["annee", "indicateur", "pays_d_echange", "type_de_viande", "periodicite"])

In [ ]:
data_path = Path("../data")
new_df = pd.read_csv(data_path / "importation.csv", sep=",", decimal=".")
new_df

In [ ]:
new_df.info()

In [ ]:
new_df.iloc[0]["annee"]